# Task 1: Sales & Demand Forecasting
## CIN: FIT/JUN26/ML9423 | Author: Abdi Negash

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/sales.csv', sep='\t')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['Year'] = df['date'].dt.year
df['Month'] = df['date'].dt.month
df['Day'] = df['date'].dt.day
df['DayOfWeek'] = df['date'].dt.dayofweek
df['DayOfYear'] = df['date'].dt.dayofyear

df_filtered = df[(df['store_nbr'] == 1) & (df['family'] == 'GROCERY I')].copy()
df_filtered = df_filtered.sort_values('date')
print(f'Filtered: {len(df_filtered)} rows')

In [ ]:
features = ['Year', 'Month', 'Day', 'DayOfWeek', 'DayOfYear', 'onpromotion']
X = df_filtered[features]
y = df_filtered['sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

def eval_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

lr_mae, lr_rmse, lr_r2 = eval_model(y_test, lr_pred)
rf_mae, rf_rmse, rf_r2 = eval_model(y_test, rf_pred)

print('Linear Regression: MAE={:.2f}, RMSE={:.2f}, R2={:.4f}'.format(lr_mae, lr_rmse, lr_r2))
print('Random Forest: MAE={:.2f}, RMSE={:.2f}, R2={:.4f}'.format(rf_mae, rf_rmse, rf_r2))

In [ ]:
best = rf if rf_r2 > lr_r2 else lr
joblib.dump(best, '../models/sales_forecast_model.pkl')
print('Best model saved!')